# Data Acquisition (XJTU Downsampling)
**Goal:** Aggregate and downsample the raw XJTU vibration data so that 1 CSV file = 1 Bearing (where 1 row represents 1 minute of operation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Define input and output paths (Modify as necessary)
RAW_DATA_PATH = r"../Raw_Data/XJTU-SY" 
OUTPUT_PATH = r"../Processed_Data/Downsampled"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dataset Specifics
# XJTU: 25.6 kHz, ~1.28 sec snapshot (32,768 rows) every 1 minute.
# We will simply load and structure these files, potentially extracting 
# baseline representations (like saving it for feature extraction later).
ROWS_PER_FILE = 32768

print(f"Raw Data Path: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")

In [ ]:
def process_and_combine_bearing_data():
    """
    Loops through Condition folders -> Bearing folders -> .csv files.
    Aggregates the raw 32,768 row arrays of vibration data into either simpler arrays
    or structures them for feature extraction later. Here we simply load and save the whole 
    array of Horiz/Vert vibration as lists or precomputed basic statistics per minute 
    (though the feature extraction pipeline does the heavy lifting).
    
    Given the constraint: 1 CSV file = 1 Bearing (1 row = 1 minute of operation, which equals 1 raw CSV file)
    """
    if not os.path.exists(RAW_DATA_PATH):
        print(f"Warning: Raw data path '{RAW_DATA_PATH}' does not exist. Please double-check.")
        return

    # List Condition Folders (e.g., Condition_1)
    condition_folders = [f for f in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, f))]
    
    for condition in condition_folders:
        cond_path = os.path.join(RAW_DATA_PATH, condition)
        bearing_folders = [f for f in os.listdir(cond_path) if os.path.isdir(os.path.join(cond_path, f))]
        
        for bearing in tqdm(bearing_folders, desc=f"Processing {condition}"):
            bearing_path = os.path.join(cond_path, bearing)
            csv_files = sorted(glob.glob(os.path.join(bearing_path, "*.csv")))
            
            # Prepare a list to collect data
            bearing_data_list = []
            
            # Extract minute information from sorted CSVs
            for minute_idx, file in enumerate(csv_files):
                # Load the raw minute file
                # Assuming XJTU has columns ['Horizontal_vibration_signals', 'Vertical_vibration_signals']
                df_min = pd.read_csv(file)
                
                # We can store the raw arrays as JSON strings or flatten them, 
                # but to be practical for python pandas loading in step 2:
                # We'll calculate a simple proxy downsampling or pass it cleanly.
                # Since Notebook 2 requires raw signal to compute features using CrossDomainFeatureExtractor,
                # we'll save the arrays as space/comma separated strings or numpy arrays.
                
                # Extract signals
                if 'Horizontal_vibration_signals' in df_min.columns:
                    h_sig = df_min['Horizontal_vibration_signals'].values
                    v_sig = df_min['Vertical_vibration_signals'].values
                else:
                    # Generic fallback if columns differ
                    h_sig = df_min.iloc[:, 0].values
                    v_sig = df_min.iloc[:, 1].values if df_min.shape[1] > 1 else np.zeros_like(h_sig)
                
                # Package 1 row per minute
                minute_record = {
                    'Minute': minute_idx + 1,
                    'H_Signal': ",".join(map(str, h_sig)), # Store as delimited string for safe CSV saving
                    'V_Signal': ",".join(map(str, v_sig))
                }
                bearing_data_list.append(minute_record)
            
            # Convert to DataFrame and Save
            if bearing_data_list:
                final_df = pd.DataFrame(bearing_data_list)
                output_filename = os.path.join(OUTPUT_PATH, f"{bearing}_downsampled.csv")
                final_df.to_csv(output_filename, index=False)
                print(f"Saved: {output_filename} (Rows: {len(final_df)})")

# Execute the pipeline
# Uncomment when raw data is in place
# process_and_combine_bearing_data()